# Part 1: Saving and restoring a model

Training takes time, so you save what training produced. In PyTorch the thing to save is the **state dict**: a dictionary that maps each layer's name to its tensor of numbers. The class definition stays in your code.

In [1]:
import torch
from torch import nn

from churn_data import make_splits
from churn_net import ChurnNet

splits = make_splits(42)
x, y = splits.train.x, splits.train.y

torch.manual_seed(0)
model = ChurnNet(splits.n_features)
for name, tensor in model.state_dict().items():
    print(f"{name:16s} {tuple(tensor.shape)}")
print("parameters:", sum(p.numel() for p in model.parameters()))

layers.0.weight  (16, 9)
layers.0.bias    (16,)
layers.2.weight  (16, 16)
layers.2.bias    (16,)
layers.4.weight  (1, 16)
layers.4.bias    (1,)
parameters: 449


## Save the weights and load them into a fresh model

A new `ChurnNet` starts with different random numbers, so its predictions differ. After `load_state_dict` they match exactly. `weights_only=True` makes `torch.load` accept only tensors and plain Python containers, which is the safe habit.

In [2]:
torch.save(model.state_dict(), "weights.pt")

fresh = ChurnNet(splits.n_features)
print("before loading:", torch.equal(model(x[:5]), fresh(x[:5])))

fresh.load_state_dict(torch.load("weights.pt", weights_only=True))
print("after loading: ", torch.equal(model(x[:5]), fresh(x[:5])))

before loading: False
after loading:  True


## A checkpoint is more than the weights

To **resume** training you also need the optimizer's memory. Adam keeps a running average of each gradient (`exp_avg`) and of its square (`exp_avg_sq`), and a step counter. Lose those and training restarts with an empty memory, which is not the same as continuing.

The test below trains for 10 steps straight, then trains 5 steps, saves, loads, and trains 5 more, with and without the optimizer state. It uses the whole training set as one batch so that nothing is random.

In [3]:
loss_fn = nn.BCEWithLogitsLoss()


def start():
    torch.manual_seed(1)
    net = ChurnNet(splits.n_features)
    return net, torch.optim.Adam(net.parameters(), lr=3e-3)


def steps(net, optimizer, count):
    net.train()
    for _ in range(count):
        optimizer.zero_grad()
        loss_fn(net(x), y).backward()
        optimizer.step()
    return loss_fn(net(x), y).item()


net, opt = start()
straight = steps(net, opt, 10)

net, opt = start()
steps(net, opt, 5)
torch.save({"model": net.state_dict(), "optimizer": opt.state_dict(), "epoch": 5}, "ck.pt")
checkpoint = torch.load("ck.pt", weights_only=True)

with_state, opt_a = start()
with_state.load_state_dict(checkpoint["model"])
opt_a.load_state_dict(checkpoint["optimizer"])

weights_only, opt_b = start()
weights_only.load_state_dict(checkpoint["model"])

print(f"10 steps straight        {straight:.6f}")
print(f"resume with optimizer    {steps(with_state, opt_a, 5):.6f}")
print(f"resume weights only      {steps(weights_only, opt_b, 5):.6f}")
print("optimizer state holds:", list(checkpoint["optimizer"]["state"][0]))

10 steps straight        0.613181
resume with optimizer    0.613181
resume weights only      0.612352
optimizer state holds: ['step', 'exp_avg', 'exp_avg_sq']


## Saving the whole model object is fragile

`torch.save(model, path)` pickles the class by name and location, so it breaks when you rename a file. Since PyTorch 2.6, `torch.load` also refuses it by default. Save the state dict, and rebuild the model from your own code.

In [4]:
torch.save(model, "whole_model.pt")
try:
    torch.load("whole_model.pt", weights_only=True)
except Exception as error:
    print(type(error).__name__)
    print(str(error)[:90], "...")

UnpicklingError
Weights only load failed. This file can still be loaded, to do so you have two options, [ ...


## Your turn

1. Save a checkpoint dictionary that also holds the best validation loss, then load it and print it.
2. Change the optimizer to `SGD` with `momentum=0.9`. Look at `checkpoint["optimizer"]["state"]` again. What is stored now?